# BETO Multi-Label Tag Classification - Training on GPU

Fine-tunes BETO (Spanish BERT) on synthetic story synopsis data for multi-label tag prediction.
Target: F1 Macro > 0.65

## Setup
1. Run all cells in order
2. The trained model will be saved to Google Drive
3. Upload `pytorch_model.bin` back to GitHub LFS or download locally

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install torch transformers scikit-learn numpy

In [ ]:
!git clone https://github.com/Xavi36772/-tagging-service.git
%cd -tagging-service

In [ ]:
import json, gzip, urllib.request

# Download merged dataset (train.json + val.json)
# If you have the combined files locally, upload them to Colab instead
base = 'https://raw.githubusercontent.com/Xavi36772/-tagging-service/master/dataset/'

# Generate fresh diverse dataset (faster than downloading large files)
!python generate_dataset.py --samples 5000

# Merge with existing template data (if available on GitHub)
# Otherwise just use the diverse dataset alone
print('Dataset ready!')

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    !nvidia-smi

In [ ]:
# Train from scratch on merged dataset with GPU
# 30 epochs should take ~1 hour on T4 GPU
!python train.py --epochs 30 --batch-size 32 --lr 3e-5 --data-dir dataset

In [ ]:
# Quick evaluation
!python eval_model.py

In [ ]:
import shutil, os

# Save model to Google Drive
drive_dir = '/content/drive/MyDrive/beto_model/'
os.makedirs(drive_dir, exist_ok=True)

for f in ['pytorch_model.bin', 'thresholds.npy', 'metrics.json', 'tokenizer.json', 'tokenizer_config.json']:
    src = f'model/{f}'
    if os.path.exists(src):
        shutil.copy(src, drive_dir)
        print(f'Copied {f} to Drive')

print(f'\nModel saved to: {drive_dir}')
print(f'\nFinal metrics:')
with open('model/metrics.json') as fp:
    m = json.load(fp)
print(f"  F1 Macro: {m['f1_macro']:.4f}")
print(f"  F1 Micro: {m['f1_micro']:.4f}")
print(f"  Hamming Loss: {m['hamming_loss']:.4f}")

In [ ]:
# OPTIONAL: Upload model back to GitHub
# You'll need a GitHub token with repo access

import requests

GITHUB_TOKEN = "your_github_token_here"  # Replace with your token
REPO = "Xavi36772/-tagging-service"
BRANCH = "master"

# Check if model already committed via LFS
# Simple approach: manually download pytorch_model.bin from Drive and push via git LFS locally

## Post-Training Steps

1. **Download the model** from Google Drive: `beto_model/pytorch_model.bin` (440MB)
2. **Place it in** `tagging-service/model/pytorch_model.bin`
3. **Update Railway**: Commit and push to trigger redeploy
   ```bash
   git add model/pytorch_model.bin
   git commit -m "feat: trained model with GPU, F1 Macro > 0.65"
   git push origin master
   ```
4. **Update CACHE_BUST** in Dockerfile (increment the number)
5. **Verify**: `POST /predict-tags` should show improved accuracy